# Intro to Machine Learning — Midterm Project
## Dataset: Adult Income (UCI / Kaggle)
### Dataset URL: https://www.kaggle.com/datasets/wenruliu/adult-income-dataset

**Team Members:** *(Add names here)*  
**Date:** March 2026  

---

## Project Goal
Use the Adult Income dataset to build a **binary classification model** that predicts whether a person earns more or less than \$50,000/year based on demographic and employment features.

This notebook covers:
1. Dataset Overview & Train/Test Split
2. Exploratory Data Analysis (on training data only)
3. Data Cleaning (handling `?` / missing values)
4. Scaling & Normalization
5. Encoding (Label + One-Hot)
6. Feature Engineering
7. Before & After Comparison
8. Export of Clean Dataset

---
## 1. Imports & Load Dataset

### 📌 What this cell does
This cell sets up the entire Python environment for the project by importing all required libraries:
- **pandas** — for loading and manipulating the dataset as a DataFrame
- **numpy** — for numerical operations and handling missing values
- **matplotlib / seaborn** — for creating charts and visualizations
- **scikit-learn** — for train/test splitting and preprocessing tools (scaling, encoding)

It then loads `adult_income.csv` into a raw DataFrame `df_raw`.

### ✅ Expected Output
- A printed line showing the dataset shape: **(32561, 15)** — meaning 32,561 rows and 15 columns
- A preview table showing the first 5 rows of the raw dataset with all original column names and values (including `?` placeholders)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load raw dataset
df_raw = pd.read_csv('adult_income.csv')
print(f'Dataset shape: {df_raw.shape}')
df_raw.head()

---
## 2. Train / Test Split (70% / 30%)
> **Important:** EDA and all preprocessing are performed ONLY on training data. The test set is set aside and left untouched until model evaluation.

### 📌 What this cell does
Before any analysis or cleaning, the full dataset is split into two separate sets:
- **Training set (70%)** — used for all EDA, cleaning, and preprocessing steps
- **Testing set (30%)** — saved untouched and only used later during model evaluation

The `stratify=df_raw['income']` parameter ensures both sets have the same proportion of `<=50K` vs `>50K` labels — preventing one split from being skewed. `random_state=42` makes the split reproducible so running it again always produces the same result.

### ✅ Expected Output
- Training set: **~22,792 rows (70%)**
- Testing set: **~9,769 rows (30%)**
- Class distribution showing roughly **75% <=50K** and **25% >50K** in the training set

In [ ]:
# Split BEFORE any processing — test data is never touched during EDA/cleaning
df_train, df_test = train_test_split(df_raw, test_size=0.30, random_state=42, stratify=df_raw['income'])

print(f'Training set size : {df_train.shape[0]:,} rows ({df_train.shape[0]/len(df_raw)*100:.1f}%)')
print(f'Testing  set size : {df_test.shape[0]:,} rows ({df_test.shape[0]/len(df_raw)*100:.1f}%)')

# Save test set aside — untouched
df_test_untouched = df_test.copy()
print('\nClass distribution in training set:')
print(df_train['income'].value_counts(normalize=True).round(3))

---
## 3. Exploratory Data Analysis (Training Data Only)

### 📌 What this cell does
Provides a high-level structural overview of the training dataset:
- **Shape** — confirms how many rows and columns are in the training set
- **Data types** — identifies which columns are numeric (`int64`) vs text (`object`), which tells us what preprocessing is needed
- **Basic statistics** — for each numeric column, shows count, mean, std, min, max, and quartiles

### ✅ Expected Output
- Shape: **(22792, 15)**
- 6 numeric columns and 9 text/categorical columns listed
- A statistics table showing, for example, that `age` ranges from 17–90 and `hours.per.week` averages around 40

In [ ]:
print('=== Training Set Overview ===')
print(f'Shape: {df_train.shape}')
print(f'\nData types:\n{df_train.dtypes}')
print(f'\nBasic statistics:')
df_train.describe()

### 📌 What this cell does
The raw dataset uses `?` as a placeholder for unknown/missing values rather than proper `NaN`. This cell:
1. Creates a temporary copy of the training data
2. Replaces all `?` strings with `NaN` (Python's standard missing value marker)
3. Counts and calculates the percentage of missing values per column
4. Displays only the columns that actually have missing data

This step is critical for understanding the scope of data quality issues before cleaning.

### ✅ Expected Output
A table showing 3 columns with missing values:
- `workclass` — ~1,285 missing (~5.6%)
- `occupation` — ~1,290 missing (~5.7%)
- `native.country` — ~408 missing (~1.8%)

In [ ]:
# Identify disguised missing values ('?')
print('=== Missing / ? Values in Training Set ===')
df_train_copy = df_train.copy()
df_train_copy.replace('?', np.nan, inplace=True)

missing = df_train_copy.isnull().sum()
missing_pct = (df_train_copy.isnull().sum() / len(df_train_copy) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)

### 📌 What this cell does
Produces two side-by-side charts to visually explore the training data:
1. **Income Distribution bar chart** — shows the count of `<=50K` vs `>50K` earners, revealing the class imbalance in our target variable
2. **Age Distribution histogram** — shows the spread of ages across the dataset, helping identify skewness or unusual patterns

Both charts are saved as `eda_distributions.png`.

### ✅ Expected Output
- Bar chart: roughly **17,285 people earning <=50K** and **5,507 earning >50K** — a clear 3:1 imbalance
- Histogram: age peaking around 35–45 with a right skew toward older ages

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

income_counts = df_train['income'].value_counts()
axes[0].bar(income_counts.index, income_counts.values, color=['steelblue', 'coral'])
axes[0].set_title('Target: Income Distribution (Training Set)')
axes[0].set_ylabel('Count')
for i, v in enumerate(income_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center')

# Age distribution
axes[1].hist(df_train['age'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Age Distribution (Training Set)')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Chart saved.')

### 📌 What this cell does
Creates a 3×3 grid of horizontal bar charts — one for each categorical feature. Each chart shows the top 8 most frequent values, making it easy to spot dominant categories, rare categories that may need grouping, and any remaining `?` values.

The chart grid is saved as `eda_categorical.png`.

### ✅ Expected Output
7 bar charts covering `workclass`, `education`, `marital.status`, `occupation`, `relationship`, `sex`, and `race`. The two unused subplot slots are hidden. Notable observations: `Private` dominates workclass, `HS-grad` dominates education, and `Male` is roughly twice as common as `Female`.

In [ ]:
# Categorical feature exploration
cat_cols = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'sex', 'race']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df_train[col].value_counts().head(8)
    axes[i].barh(counts.index, counts.values, color='steelblue')
    axes[i].set_title(f'{col}')
    axes[i].invert_yaxis()

# Hide unused subplots
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions (Training Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_categorical.png', dpi=100, bbox_inches='tight')
plt.show()

### 📌 What this cell does
Creates three **box plots** comparing key numeric features split by income class (`<=50K` vs `>50K`). Box plots show the median, spread, and outliers of a distribution — making it easy to see whether a feature differs meaningfully between income groups:
- **Age by Income** — do higher earners tend to be older?
- **Hours/Week by Income** — do higher earners work more hours?
- **Education Level by Income** — do higher earners have more years of education?

The chart is saved as `eda_income_features.png`.

### ✅ Expected Output
Three box plots showing that people earning `>50K` tend to be **older**, work **more hours per week**, and have **higher education levels** — confirming these are likely strong predictive features for our ML model

In [ ]:
# Income by key features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Age by income
df_train.boxplot(column='age', by='income', ax=axes[0])
axes[0].set_title('Age by Income')
axes[0].set_xlabel('Income')

# Hours per week by income
df_train.boxplot(column='hours.per.week', by='income', ax=axes[1])
axes[1].set_title('Hours/Week by Income')
axes[1].set_xlabel('Income')

# Education num by income
df_train.boxplot(column='education.num', by='income', ax=axes[2])
axes[2].set_title('Education Level by Income')
axes[2].set_xlabel('Income')

plt.suptitle('')
plt.tight_layout()
plt.savefig('eda_income_features.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 4. Data Cleaning
### 4a. Handle Missing / '?' Values
Three columns contain `?` as a disguised missing value: `workclass`, `occupation`, and `native.country`.  
We replace `?` with `NaN` then fill with the **mode** (most frequent category) of each column from the training set.

### 📌 What this cell does
Performs the core missing value cleanup on the training data:
1. Creates a working copy `df_clean` so the original split is preserved
2. Replaces all `?` strings with proper `NaN` missing values
3. For each affected column, calculates the **mode** (most frequent value) from training data
4. Fills all `NaN` entries with that mode value

Using the **mode** is appropriate for categorical columns — it fills gaps with the most common real category without inventing new data. The fill values are stored in `fill_values` so the same values can be applied to the test set later.

### ✅ Expected Output
- A preview of rows that previously had `?` values
- Three fill confirmations (e.g., `workclass` filled with `Private`, `occupation` filled with `Prof-specialty`)
- Confirmation of **0 remaining nulls** after cleaning

In [ ]:
# Work on a copy of training data
df_clean = df_train.copy()

print('=== BEFORE Cleaning ===')
print('Sample rows with ? values:')
print(df_clean[df_clean['workclass'] == '?'][['age','workclass','occupation','native.country','income']].head(5))

# Replace '?' with NaN
df_clean.replace('?', np.nan, inplace=True)

# Fill NaN with mode from training set
cols_with_missing = ['workclass', 'occupation', 'native.country']
fill_values = {}
for col in cols_with_missing:
    mode_val = df_clean[col].mode()[0]
    fill_values[col] = mode_val
    df_clean[col].fillna(mode_val, inplace=True)
    print(f'Filled "{col}" NaN with mode: "{mode_val}"')

print(f'\nRemaining nulls after cleaning: {df_clean.isnull().sum().sum()}')

### 📌 What this cell does
A **verification step** confirming the cleaning was successful. It re-checks null counts for the three previously-missing columns and displays updated value counts for `workclass` to confirm `?` no longer appears.

This kind of before/after verification is an important data engineering practice to ensure no missing values slipped through.

### ✅ Expected Output
- All three columns now show **0 nulls**
- The `workclass` value counts table no longer contains a `?` row — those entries have been reassigned to `Private`

In [ ]:
print('=== AFTER Cleaning — No More ? / NaN ===')
print(df_clean[['workclass','occupation','native.country']].isnull().sum())
print()
print('workclass value counts (? gone):')
print(df_clean['workclass'].value_counts())

---
### 4b. Scaling — Changing the Range of Data (Min-Max Scaling)
Scales numeric features to the range [0, 1]. Important for distance-based algorithms (KNN, SVM).

### 📌 What this cell does
Different numeric columns have very different value ranges — `age` goes 17–90 while `fnlwgt` goes up to 1,484,705. Without scaling, algorithms that rely on distances or magnitudes (like KNN or SVM) would incorrectly treat `fnlwgt` as far more important simply because its numbers are larger.

**Min-Max Scaling** transforms each numeric column so the minimum becomes **0** and the maximum becomes **1**, with all values proportionally in between.

Formula: `x_scaled = (x - x_min) / (x_max - x_min)`

### ✅ Expected Output
A side-by-side min/max/mean table:
- **Before**: wildly different ranges across columns (e.g., fnlwgt max = 1,484,705)
- **After**: every column has min = **0.00** and max = **1.00**, means are now fractions between 0 and 1

In [ ]:
numeric_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

print('=== BEFORE Scaling ===')
print(df_clean[numeric_cols].describe().loc[['min','max','mean']])

# Apply MinMax scaling
scaler = MinMaxScaler()
df_scaled = df_clean.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

print('\n=== AFTER Min-Max Scaling (range now 0 to 1) ===')
print(df_scaled[numeric_cols].describe().loc[['min','max','mean']])

---
### 4c. Normalization — Adjusting the Shape of the Distribution (Z-score / Standard Scaling)
Transforms features to have **mean = 0** and **standard deviation = 1**. Applied to skewed numeric features.

### 📌 What this cell does
Some columns like `capital.gain` and `capital.loss` are **heavily right-skewed** — the vast majority of people have a value of 0, with a small number having very large values. This skewness can negatively affect model performance.

**Z-score (Standard) Normalization** transforms these columns so the mean becomes **0** and standard deviation becomes **1**, pulling extreme outliers closer to the center.

Formula: `x_normalized = (x - mean) / std_dev`

This cell also produces side-by-side histograms (before in blue, after in coral) showing how the distribution shape changes.

### ✅ Expected Output
- 6 histograms (3 before, 3 after) for `capital.gain`, `capital.loss`, and `fnlwgt`
- Before: extreme spike near zero (very skewed)
- After: distribution recentered around 0 with a more balanced spread
- Printed mean values close to **0.0000** and std values close to **1.0000**

In [ ]:
# Visualize skewness BEFORE normalization
skewed_cols = ['capital.gain', 'capital.loss', 'fnlwgt']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, col in enumerate(skewed_cols):
    axes[0, i].hist(df_clean[col], bins=40, color='steelblue', edgecolor='white')
    axes[0, i].set_title(f'BEFORE: {col}')

# Apply StandardScaler (Z-score normalization)
std_scaler = StandardScaler()
df_normalized = df_scaled.copy()
df_normalized[skewed_cols] = std_scaler.fit_transform(df_clean[skewed_cols])

for i, col in enumerate(skewed_cols):
    axes[1, i].hist(df_normalized[col], bins=40, color='coral', edgecolor='white')
    axes[1, i].set_title(f'AFTER: {col} (Z-score)')

plt.suptitle('Normalization: Before vs After', fontsize=13)
plt.tight_layout()
plt.savefig('normalization_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('Mean after normalization (should be ~0):')
print(df_normalized[skewed_cols].mean().round(4))
print('Std after normalization (should be ~1):')
print(df_normalized[skewed_cols].std().round(4))

---
## 5. Encoding
### 5a. Label Encoding — Binary Categorical Columns
For columns with only 2 categories (e.g., `sex`, `income`), we use **Label Encoding** (0 / 1).

### 📌 What this cell does
ML models cannot process raw text — all data must be numeric. For columns with exactly **2 unique categories**, Label Encoding is the simplest approach: it assigns `0` to one category and `1` to the other.

Applied to:
- **`sex`** — `Female` → 0, `Male` → 1
- **`income`** — `<=50K` → 0, `>50K` → 1 (this is our **target variable**)

Label encoding is safe here because there are only 2 values — no false numeric ordering is implied.

### ✅ Expected Output
- Before: columns showing text values like `Male` and `<=50K`
- The exact encoding mapping printed for each column
- After: same columns now containing only `0` and `1`

In [ ]:
df_encoded = df_normalized.copy()

print('=== BEFORE Label Encoding ===')
print(df_encoded[['sex', 'income']].head(5))

# Label encode binary columns
le = LabelEncoder()
binary_cols = ['sex', 'income']
for col in binary_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f'  {col} mapping: {mapping}')

print('\n=== AFTER Label Encoding ===')
print(df_encoded[['sex', 'income']].head(5))

---
### 5b. One-Hot Encoding — Multi-Category Columns
For categorical columns with more than 2 categories, we use **One-Hot Encoding** to avoid implying ordinal relationships.

### 📌 What this cell does
For columns with **more than 2 categories** (like `occupation` with 14 values), label encoding would incorrectly imply a numeric ordering. **One-Hot Encoding** instead creates a new binary (0/1) column for each unique category.

For example, `workclass` gets split into separate columns: `workclass_Local-gov`, `workclass_Private`, etc. Each row gets a `1` in the column matching its value and `0` everywhere else.

`drop_first=True` removes one redundant column per feature to avoid **multicollinearity**.

Applied to 7 columns: `workclass`, `education`, `marital.status`, `occupation`, `relationship`, `race`, `native.country`.

### ✅ Expected Output
- **Before**: 15 columns containing text
- **After**: ~100 columns — all numeric 0/1 values
- A printed sample of the newly created column names (e.g., `workclass_Local-gov`, `education_Bachelors`)

In [ ]:
ohe_cols = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'native.country']

print(f'=== BEFORE One-Hot Encoding ===')
print(f'Shape: {df_encoded.shape}')
print(df_encoded[ohe_cols].head(3))

# One-Hot Encode
df_ohe = pd.get_dummies(df_encoded, columns=ohe_cols, drop_first=True)

print(f'\n=== AFTER One-Hot Encoding ===')
print(f'Shape: {df_ohe.shape} (columns expanded from {df_encoded.shape[1]} to {df_ohe.shape[1]})')
print('New columns added (sample):')
new_cols = [c for c in df_ohe.columns if any(c.startswith(cat) for cat in ohe_cols)]
print(new_cols[:15], '...')

---
## 6. Feature Engineering
Creating new meaningful features from existing ones to improve predictive power.

### 📌 What this cell does
Feature engineering creates **new columns** derived from existing ones that may give the model stronger predictive signal. Three new features are created:

1. **`capital_net`** — subtracts `capital.loss` from `capital.gain` for a single net capital figure. Someone with high gains and no losses is financially very different from someone with both — this captures that nuance in one number.

2. **`age_group`** — bins continuous age into 5 life/career stages: `Young` (≤25), `EarlyCareer` (26–35), `MidCareer` (36–50), `LateCareer` (51–65), `Senior` (65+). These groups may have distinct income patterns that a raw age number doesn't capture as cleanly. Labels are then integer-encoded.

3. **`is_high_hours`** — a binary flag (0/1) for whether someone works more than 45 hours per week. High-hour workers may correlate strongly with higher income, and isolating it as a simple yes/no feature helps certain model types.

### ✅ Expected Output
- Summary stats for `capital_net` showing its value range
- Value counts for `age_group` (0–4) showing distribution across life stages
- Value counts for `is_high_hours` (0 = normal, 1 = high)
- Final shape: **(22792, 103)** — original features + one-hot columns + 3 new engineered features

In [ ]:
df_feat = df_ohe.copy()

# NOTE: age and hours.per.week are scaled (0-1). Use original df_clean for engineering then re-scale.
# For clarity, we reconstruct from original training values:
df_feat_check = df_clean.copy()
df_feat_check.replace('?', np.nan, inplace=True)
for col in cols_with_missing:
    df_feat_check[col].fillna(fill_values[col], inplace=True)

# Feature 1: capital_net = capital.gain - capital.loss
df_feat['capital_net'] = df_feat_check['capital.gain'] - df_feat_check['capital.loss']
print('Feature 1 — capital_net (capital gain minus capital loss):')
print(df_feat['capital_net'].describe())

# Feature 2: age_group — bin age into life stages
df_feat['age_group'] = pd.cut(
    df_feat_check['age'],
    bins=[0, 25, 35, 50, 65, 100],
    labels=['Young', 'EarlyCareer', 'MidCareer', 'LateCareer', 'Senior']
)
df_feat['age_group'] = LabelEncoder().fit_transform(df_feat['age_group'].astype(str))
print('\nFeature 2 — age_group value counts:')
print(df_feat['age_group'].value_counts())

# Feature 3: is_high_hours — binary flag for people working more than 45 hrs/week
df_feat['is_high_hours'] = (df_feat_check['hours.per.week'] > 45).astype(int)
print('\nFeature 3 — is_high_hours (>45 hrs/week):')
print(df_feat['is_high_hours'].value_counts())

print(f'\nFinal dataset shape after feature engineering: {df_feat.shape}')

---
## 7. Before & After Summary Comparison

### 📌 What this cell does
Produces a **master transformation summary table** tracking how the dataset changed at every pipeline stage — capturing row count, column count, and missing value count after each step.

This is the core **before and after evidence** required by the assignment, showing the full journey from raw data to ML-ready dataset in one clear table.

### ✅ Expected Output
An 8-row summary table showing:
- Row count stays constant at **22,792** throughout (no data is dropped)
- Columns grow from **15 → 103** as encoding and feature engineering expand the feature space
- Missing values drop from **~2,983 → 0** after the cleaning step and stay at 0 for all subsequent stages

In [ ]:
summary = {
    'Stage': [
        'Raw Dataset',
        'After Train/Test Split (train)',
        'After Cleaning (? -> mode)',
        'After Scaling (MinMax)',
        'After Normalization (Z-score)',
        'After Label Encoding',
        'After One-Hot Encoding',
        'After Feature Engineering'
    ],
    'Rows': [
        df_raw.shape[0],
        df_train.shape[0],
        df_clean.shape[0],
        df_scaled.shape[0],
        df_normalized.shape[0],
        df_encoded.shape[0],
        df_ohe.shape[0],
        df_feat.shape[0]
    ],
    'Columns': [
        df_raw.shape[1],
        df_train.shape[1],
        df_clean.shape[1],
        df_scaled.shape[1],
        df_normalized.shape[1],
        df_encoded.shape[1],
        df_ohe.shape[1],
        df_feat.shape[1]
    ],
    'Missing Values': [
        df_raw.replace('?', np.nan).isnull().sum().sum(),
        df_train.replace('?', np.nan).isnull().sum().sum(),
        df_clean.isnull().sum().sum(),
        df_scaled.isnull().sum().sum(),
        df_normalized.isnull().sum().sum(),
        df_encoded.isnull().sum().sum(),
        df_ohe.isnull().sum().sum(),
        df_feat.isnull().sum().sum()
    ]
}

summary_df = pd.DataFrame(summary)
print('=== Dataset Transformation Summary ===')
print(summary_df.to_string(index=False))

### 📌 What this cell does
Produces a **3-panel visual** summarizing the most important before/after transformations:
1. **Age — Raw**: histogram of original age values (range 17–90)
2. **Age — After MinMax Scaling**: same distribution shape, but x-axis now spans [0, 1] — proving scaling changes the range without changing the shape
3. **Target Variable Encoded**: bar chart showing the final 0/1 income labels, confirming class imbalance is preserved after encoding

Saved as `before_after_comparison.png`.

### ✅ Expected Output
Three side-by-side charts. The first two histograms should look **identical in shape** but have different x-axis scales — this is the key visual proof that MinMax scaling only changes the range, not the distribution structure.

In [ ]:
# Visual comparison: age column before and after scaling
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Before scaling
axes[0].hist(df_clean['age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Age - Raw (Training)')
axes[0].set_xlabel('Age')

# After MinMax scaling
axes[1].hist(df_scaled['age'], bins=30, color='green', edgecolor='white')
axes[1].set_title('Age - After MinMax Scaling')
axes[1].set_xlabel('Scaled Value [0,1]')

# Income class balance
income_map = {'<=50K': 0, '>50K': 1}
income_encoded = df_train['income'].map(income_map)
axes[2].bar(['<=50K (0)', '>50K (1)'], income_encoded.value_counts().sort_index().values,
            color=['steelblue', 'coral'])
axes[2].set_title('Target Variable (income) - Encoded')
axes[2].set_ylabel('Count')

plt.suptitle('Before vs After Transformations', fontsize=13)
plt.tight_layout()
plt.savefig('before_after_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 8. Export Clean Dataset

### 📌 What this cell does
Saves the two final datasets as CSV files for submission and future use in the final exam:
1. **`adult_income_clean_train.csv`** — the fully processed training set with all cleaning, scaling, normalization, encoding, and feature engineering applied. This is ready to be fed directly into an ML model.
2. **`adult_income_test_raw.csv`** — the untouched 30% test set in its original raw form, preserved so it can be processed and evaluated consistently when the model is tested.

### ✅ Expected Output
- Two confirmation messages with file names and shapes
- Clean training set: **(22792, 103)**
- Raw test set: **(9769, 15)** — still in original form with 15 columns
- A preview table showing the first 3 rows of the clean training data — all numeric, no text

In [ ]:
# Export clean training data
df_feat.to_csv('adult_income_clean_train.csv', index=False)
print(f'Clean training dataset saved: adult_income_clean_train.csv')
print(f'Shape: {df_feat.shape}')

# Save untouched test set
df_test_untouched.to_csv('adult_income_test_raw.csv', index=False)
print(f'\nUntouched test dataset saved: adult_income_test_raw.csv')
print(f'Shape: {df_test_untouched.shape}')

print('\nFirst 3 rows of clean training data:')
df_feat.head(3)

---
## 9. ML Problem Proposal

### Problem Statement
**Binary Classification**: Predict whether an individual's annual income exceeds $50,000 based on demographic and employment attributes.

### Target Variable
- `income` — encoded as `0` (<=50K) and `1` (>50K)

### Feature Variables
After preprocessing, all 15 original features plus 3 engineered features (`capital_net`, `age_group`, `is_high_hours`) are used as input.

### Suggested Models
1. **Logistic Regression** — baseline interpretable model
2. **Random Forest Classifier** — handles non-linear patterns, feature importance
3. **Gradient Boosting (XGBoost)** — higher accuracy potential

### Evaluation Metrics
- **Accuracy** — overall correct predictions
- **Precision / Recall / F1** — important because classes are imbalanced (~75% <=50K vs ~25% >50K)
- **ROC-AUC** — measures discrimination ability

### Class Imbalance Note
The dataset has a ~3:1 class imbalance. Techniques like **SMOTE** (oversampling) or **class_weight='balanced'** may be applied during model training.

### Real-World Application
This model could assist in financial services for credit risk assessment, targeted marketing, or social program eligibility screening — areas where income estimation is a proxy for financial behavior.

---
## 10. Reflection Journal
> **Instructions:** Each team member should fill in their section below describing their individual contribution to this project. This section directly affects individual grades (+10% / -10% from the base grade).

---

### 👤 Team Member 1 — *[Full Name]*

**Contribution:**  
*(Describe what you personally worked on — e.g., data loading, EDA, visualizations, cleaning strategy, etc.)*

**What I learned:**  
*(Reflect on new techniques or insights gained from this project.)*

**Challenges faced:**  
*(Describe any difficulties and how you resolved them.)*

---

### 👤 Team Member 2 — *[Full Name]*

**Contribution:**  
*(Describe what you personally worked on — e.g., scaling/normalization, encoding, feature engineering, etc.)*

**What I learned:**  
*(Reflect on new techniques or insights gained from this project.)*

**Challenges faced:**  
*(Describe any difficulties and how you resolved them.)*

---

### 👤 Team Member 3 — *[Full Name]*

**Contribution:**  
*(Describe what you personally worked on — e.g., ML proposal, before/after comparison, dataset export, GitHub setup, etc.)*

**What I learned:**  
*(Reflect on new techniques or insights gained from this project.)*

**Challenges faced:**  
*(Describe any difficulties and how you resolved them.)*

---

> **Add or remove team member sections as needed.**